Belén Quiroga - Financial Data Scientist 📍 Argentina.

This Python script has some of the most commonly used technical indicators.
Simply change the asset ticker in:
TICKER = "XXX"
and you're good to go.

Feedback, suggestions, and improvements are always welcome! 🚀

If you use Binance, you can sign up using this link to get a 10% discount on your trading fees:
https://accounts.binance.com/es/register?ref=ACJGECMG





In [1]:
!pip install backtrader #Install Backtrader
!pip install yfinance #Install yfinance

from datetime import datetime
import pandas as pd
import yfinance as yf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 11.6 MB/s eta 0:00:00


In [2]:

# Ticker
TICKER = 'AAPL'

# Parameters
STOCK_DATA = yf.download(
    TICKER,
    start='2026-01-30',
    end='2026-07-30',
    interval='1d',   # '1d' (Daily), '1wk' (Weekly), '1h' (Hourly)
    progress=False,
    auto_adjust=True
)

if isinstance(STOCK_DATA.columns, pd.MultiIndex):
    STOCK_DATA.columns = STOCK_DATA.columns.get_level_values(0)


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SPY500"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SPY500']: YFTzMissingError('possibly delisted; no timezone found')


In [3]:
import numpy as np

# 1. EMA Calculation
STOCK_DATA['EMA200'] = STOCK_DATA['Close'].ewm(span=200, adjust=False).mean()
STOCK_DATA['EMA50']  = STOCK_DATA['Close'].ewm(span=50, adjust=False).mean()

# 2. Bollinger Bands
bb_middle = STOCK_DATA['Close'].rolling(window=20).mean()
bb_std = STOCK_DATA['Close'].rolling(window=20).std()
STOCK_DATA['BB_High'] = bb_middle + (2 * bb_std)
STOCK_DATA['BB_Low']  = bb_middle - (2 * bb_std)

# 3. RSI
delta = STOCK_DATA['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
STOCK_DATA['RSI'] = 100 - (100 / (1 + (gain / loss)))

# 4. MACD
STOCK_DATA['MACD'] = STOCK_DATA['Close'].ewm(span=12, adjust=False).mean() - STOCK_DATA['Close'].ewm(span=26, adjust=False).mean()
STOCK_DATA['MACD_Sign'] = STOCK_DATA['MACD'].ewm(span=9, adjust=False).mean()
STOCK_DATA['MACD_Hist'] = STOCK_DATA['MACD'] - STOCK_DATA['MACD_Sign']

# 5. VWAP
typical_price = (STOCK_DATA['High'] + STOCK_DATA['Low'] + STOCK_DATA['Close']) / 3
STOCK_DATA['VWAP'] = (typical_price * STOCK_DATA['Volume']).cumsum() / STOCK_DATA['Volume'].cumsum()

# 6. Long/Short
entry_price = STOCK_DATA['Close'].iloc[-1]
#take_profit = entry_price * 1.10
#stop_loss   = entry_price * 0.90

# Set it MANUALLY
take_profit = 236
stop_loss   = 193

risk = entry_price - stop_loss
reward = take_profit - entry_price
risk_reward_ratio = reward / risk
pct_gain = (reward / entry_price) * 100
pct_loss = (risk / entry_price) * 100

# 7. Institutional Volume (2.5x standard 20-day mean)
STOCK_DATA['Vol_Avg'] = STOCK_DATA['Volume'].rolling(window=20).mean()
climax_filter = STOCK_DATA['Volume'] > (STOCK_DATA['Vol_Avg'] * 2.5)
climax_data = STOCK_DATA[climax_filter].copy()

# Map color signals
if not climax_data.empty:
    climax_data['Marker_Color'] = np.where(climax_data['Close'] >= climax_data['Open'], "rgba(0, 200, 100, 0.9)", "rgba(255, 50, 50, 0.9)")
    climax_data['Alert_Type'] = np.where(climax_data['Close'] >= climax_data['Open'], "Institutional Accumulation (Buy)", "Institutional Capitulation (Sell)")

print("todo piola")

IndexError: single positional indexer is out-of-bounds

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Graphic
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.03, row_heights=[0.60, 0.20, 0.20],
    specs=[[{"secondary_y": True}], [{"secondary_y": False}], [{"secondary_y": False}]]
)

# 2. Main Price Panel & Indicators
fig.add_trace(go.Candlestick(x=STOCK_DATA.index, open=STOCK_DATA['Open'], high=STOCK_DATA['High'], low=STOCK_DATA['Low'], close=STOCK_DATA['Close'], name=TICKER))
fig.add_trace(go.Scatter(x=STOCK_DATA.index, y=STOCK_DATA['EMA200'], line=dict(color='red'), name='EMA 200'))
fig.add_trace(go.Scatter(x=STOCK_DATA.index, y=STOCK_DATA['EMA50'], line=dict(color='green'), name='EMA 50'))
fig.add_trace(go.Scatter(x=STOCK_DATA.index, y=STOCK_DATA['VWAP'], line=dict(color='orange', width=2), name='VWAP'))
fig.add_trace(go.Bar(x=STOCK_DATA.index, y=STOCK_DATA['Volume'], name='Volume', opacity=0.2), secondary_y=True)


# Bollinger
fig.add_trace(go.Scatter(x=STOCK_DATA.index, y=STOCK_DATA['BB_High'], line=dict(color='gray', dash='dash'), name='Bollinger Alta'), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=STOCK_DATA.index, y=STOCK_DATA['BB_Low'], line=dict(color='gray', dash='dash'), name='Bollinger Baja'), row=1, col=1, secondary_y=False)

# 3. Forecasting (LONG/SHORT)
fig.add_trace(go.Scatter(
    x=[STOCK_DATA.index, STOCK_DATA.index[-1], STOCK_DATA.index[-1], STOCK_DATA.index], y=[entry_price, entry_price, take_profit, take_profit],
    fill="toself", fillcolor="rgba(0, 255, 0, 0.12)", line=dict(color="green", width=1, dash="dash"),
    name="Long Target", legendgroup="Forecasting",
    visible="legendonly"  # <--- HIDES BY DEFAULT
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[STOCK_DATA.index, STOCK_DATA.index[-1], STOCK_DATA.index[-1], STOCK_DATA.index], y=[stop_loss, stop_loss, entry_price, entry_price],
    fill="toself", fillcolor="rgba(255, 0, 0, 0.12)", line=dict(color="red", width=1, dash="dash"),
    name="Long Stop", legendgroup="Forecasting",
    visible="legendonly"  # <--- HIDES  BY DEFAULT
), row=1, col=1)

fig.add_annotation(
    xref="paper", yref="paper", x=0.01, y=0.98, showarrow=False, align="left", bgcolor="rgba(255, 255, 255, 0.85)", bordercolor="gray", borderwidth=1,
    text=(f"<b>AUTOMATED LONG METRICS</b><br>Target Price: {take_profit:.2f} (+{pct_gain:.2f}%)<br>Entry Price: {entry_price:.2f}<br>Stop Loss: {stop_loss:.2f} (-{pct_loss:.2f}%)<br>Risk/Reward Ratio: {risk_reward_ratio:.2f}")
)

# 4. FUNDAMENTAL EVENTS
chart_start = STOCK_DATA.index.min().strftime('%Y-%m-%d')
chart_end = STOCK_DATA.index.max().strftime('%Y-%m-%d')
ticker_object = yf.Ticker(TICKER)
event_y_position = STOCK_DATA['Low'].min() * 0.98

ev_st = dict(mode="markers+text", legendgroup="Corporate Events", textposition="middle center", textfont=dict(color="white", size=9, weight="bold"))

# Dividends
try:
    divs = ticker_object.dividends
    if not divs.empty:
        divs.index = divs.index.tz_localize(None) if divs.index.tz is not None else divs.index
        v_divs = divs[(divs.index >= chart_start) & (divs.index <= chart_end)]
        if not v_divs.empty:
            fig.add_trace(go.Scatter(x=v_divs.index, y=[event_y_position] * len(v_divs), mode="markers+text", name="Dividends", legendgroup="Corporate Events", marker=dict(color="rgba(0, 150, 255, 0.85)", size=13), text=["D"] * len(v_divs), textposition="middle center", textfont=dict(color="white", size=9, weight="bold"), customdata=v_divs.values, hovertemplate="<b>Dividend Payment</b><br>Date: %{x|%Y-%m-%d}<br>Amount: $%{customdata:.2f}<extra></extra>"), row=1, col=1)
except Exception:
    pass

# Earnings
try:
    earn = ticker_object.get_earnings_dates(limit=20)
    if earn is not None and not earn.empty:
        earn.index = earn.index.tz_localize(None) if earn.index.tz is not None else earn.index
        v_earn = earn[(earn.index >= chart_start) & (earn.index <= chart_end)]
        if not v_earn.empty:
            fig.add_trace(go.Scatter(x=v_earn.index, y=[event_y_position] * len(v_earn), mode="markers+text", name="Earnings", legendgroup="Corporate Events", marker=dict(color="rgba(255, 165, 0, 0.9)", size=13), text=["E"] * len(v_earn), textposition="middle center", textfont=dict(color="white", size=9, weight="bold"), customdata=np.stack((v_earn['EPS Estimate'].fillna(0), v_earn['Reported EPS'].fillna(0)), axis=-1), hovertemplate="<b>Earnings Report (E)</b><br>Date: %{x|%Y-%m-%d}<br>Est. EPS: %{customdata:.2f}<br>Reported EPS: %{customdata:.2f}<extra></extra>"), row=1, col=1)
except Exception:
    pass
# 5. Volume
if not climax_data.empty:
    fig.add_trace(go.Scatter(
        x=climax_data.index, y=climax_data['High'] * 1.015, mode="markers", name="Volume Climax", legendgroup="Market Alerts",
        marker=dict(color=climax_data['Marker_Color'], size=10, symbol="circle-open", line=dict(width=3)),
        text=[f"{v/1e6:.1f}M" for v in climax_data['Volume']], customdata=climax_data['Alert_Type'],
        hovertemplate="<b>Vol Climax: %{customdata}</b><br>Volume: %{text}<extra></extra>"
    ), row=1, col=1)

# 6. RSI & MACD Indicators
fig.add_trace(go.Scatter(x=STOCK_DATA.index, y=STOCK_DATA['RSI'], line=dict(color='purple'), name='RSI'), row=2, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="gray", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="gray", row=2, col=1)

fig.add_trace(go.Scatter(x=STOCK_DATA.index, y=STOCK_DATA['MACD'], line=dict(color='blue'), name='MACD'), row=3, col=1)
fig.add_trace(go.Scatter(x=STOCK_DATA.index, y=STOCK_DATA['MACD_Sign'], line=dict(color='orange'), name='Señal'), row=3, col=1)
fig.add_trace(go.Bar(x=STOCK_DATA.index, y=STOCK_DATA['MACD_Hist'], name='Hist', opacity=0.5, marker_color='gray'), row=3, col=1)
fig.add_hline(y=0, line_dash="solid", line_color="black", row=3, col=1)

# 7. Layout
fig.update_layout(title=f"Stock: {TICKER}", xaxis_rangeslider_visible=False, height=900)
fig.update_yaxes(title_text="Price", secondary_y=False, row=1, col=1)
fig.update_yaxes(title_text="Volume", secondary_y=True, row=1, col=1, showgrid=False)
fig.update_yaxes(title_text="RSI", range=[0, 100], row=2, col=1)
fig.update_yaxes(title_text="MACD", row=3, col=1)
fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]) # weekends
    ]
)
fig.update_layout(hovermode="x")
fig.update_traces(hoverinfo="skip")
fig.data[0].update(hoverinfo="all")


In [ ]:
#@title Config
import IPython

# 1. Cursor Interactive
fig.update_layout(hovermode="closest")
fig.update_traces(hoverinfo="all")

# 2. JavaScript Real-Time Price Tracking Engine
js_code = """<script>function _tv() { var gd = document.getElementsByClassName('plotly-graph-div')[0]; if(!gd) return setTimeout(_tv, 200); var lbl = document.createElement('div'); Object.assign(lbl.style, {position:'absolute', right:'45px', backgroundColor:'#2a2e39', color:'#d1d4dc', padding:'3px 6px', borderRadius:'3px', fontSize:'11px', zIndex:'1000', display:'none', pointerEvents:'none'}); gd.appendChild(lbl); gd.addEventListener('mousemove', (e) => { var ya = gd._fullLayout.yaxis, box = gd.getBoundingClientRect(), yM = e.clientY - box.top, p = ya.p2c(yM - gd._fullLayout.margin.t); if (p >= ya.range[0] && p <= ya.range[1]) { lbl.style.top = (yM - 10) + 'px'; lbl.innerHTML = p.toFixed(2); lbl.style.display = 'block'; } else lbl.style.display = 'none'; }); gd.addEventListener('mouseleave', () => lbl.style.display = 'none'); } setTimeout(_tv, 500);</script>"""
IPython.display.display(IPython.display.HTML(js_code))

# 3. Chart  with Drawing Tools
fig.show(config={'modeBarButtonsToAdd': ['drawline', 'drawopenpath', 'drawrect', 'eraseshape']})